
# Part 1 到底在测什么、怎么做

## 一、先回答你的问题：要不要手写 model？

**不是必须，但值得做一个"轻量版"，并且放在第一轮跑通之后。**

你的迷茫不是缺 A1 的代码，而是还没有"一次测量长什么样"的心智模型。仓库里已经有 staff 提供的 `cs336_basics`，Part 1 的所有实验都能直接用它。A1 里的 BPE、tokenizer、训练循环，A2 完全用不到。

A2 真正用到的 A1 部分只有：

| 组件 | 在 A2 里的用途 |
|---|---|
| `BasicsTransformerLM` | Part 1 的被测对象，Part 3/4 的训练模型 |
| `AdamW` | 测 optimizer step，Part 4 的分片对象 |
| `cross_entropy` | 算 loss，触发 backward |
| attention 函数（`scaled_dot_product_attention`） | Part 1 打 NVTX 标、Part 2 替换成 FlashAttention |

模块名以你仓库里的为准。

建议这样安排：

1. **先用 staff 的 model 把 3.2 的第一格跑出来**（见下文），不要被手写拖住。
2. 然后**读一遍 `model.py`，写一个约 100 行的 `MiniTransformerLM`**，包含 RMSNorm、RoPE、因果多头注意力、SwiGLU。用 `load_state_dict` 把 staff 权重载进来，对比 logits 是否 `allclose`，通过就算完成。
3. 这一步的目的是看清 matmul、softmax、elementwise 各在哪里，后面 nsys 里才能把 kernel 对应到代码。预计 3 到 5 小时。

如果时间紧，只做"带注释的通读"也行：对每个模块标出它产生哪些 kernel。

## 二、Part 1 的本质：一个被测对象，三个问题

**被测对象**：一个随机初始化的 Transformer LM，喂随机 token（`torch.randint`），不需要数据集和 tokenizer。

**要回答的三个问题**：

| 问题 | 对应实验 | 产出 |
|---|---|---|
| 每个阶段（forward / backward / optimizer）花多久，随模型和 ctx 怎么变？ | 3.2、3.3 | 计时表 |
| 时间具体花在哪些 kernel 上？ | 3.4 nsys | kernel 排名 |
| 显存花在哪里？混合精度改变了什么？ | 3.5、3.8 | 显存表、对比表 |

整个 Part 1 就是把同一个训练步拆开，分别看时间和显存。

### Step 1：先测出这块卡的"天花板"

在测模型之前，先用一个大矩阵乘法测出这块卡实际能跑多快，后面所有模型数字都拿它当参照。

- 测 `8192×8192` 的 matmul，分别在 fp32（TF32 关）、fp32（TF32 开）、bf16 下，算出 TFLOPs/s（FLOPs = 2·n³）。
- 这三个数同时回答了你文档里 3.1 那条 TF32 的问题：fp32 基线用哪种设置，就在这里确定并记下来。

- 填好的表

| 项目 | 卡 0 | 卡 1 | 采用值 |
|---|---|---|---|
| P_fp32（TF32 关，n=8192 / 16384） | 32.84 / 33.04 | 33.69 / 33.93 | **33.4 TFLOPs/s** |
| P_tf32（TF32 开） | 48.84 / 49.74 | 48.72 / 49.57 | **49.2** |
| P_bf16 | 92.36 / 98.41 | 95.03 / 98.74 | **96.1**（±3%） |
| BW（memcopy，读+写） | 768.0 | 769.6 | **769 GB/s** |
| 功耗墙 | 300 W | 300 W | 300 W |
| 稳态 SM 频率（fp32 关 / TF32 开 / bf16） | 约 2540 / 2812 / 2795 MHz | 约2650/2800/2800 | 记录即可 |

倍数关系可以写进报告：bf16 / fp32 ≈ 2.9×，TF32 / fp32 ≈ 1.5×，bf16 / TF32 ≈ 2.0×。memcopy 769 GB/s 约为标称带宽 896 GB/s 的 86%（标称值请以官方规格为准）。

**可选的验证**：确认降频原因的话，可以在跑 fp32 阶段时执行 `nvidia-smi -q -d PERFORMANCE`，看 "Clocks Event Reasons" 里是否有 `SW Power Cap: Active`。

- 决定：fp32 基线 = TF32 关

后面所有"预期时间"使用：

| 精度 | P | 用于 |
|---|---|---|
| fp32 基线 | 33.4 TFLOPs/s | 3.2、3.3 的 fp32 部分 |
| bf16 autocast | 96.1 TFLOPs/s | 3.5 混合精度对比 |
| 访存 | 769 GB/s | optimizer step |

在 3.1 的"TF32 设置"栏填：`torch.set_float32_matmul_precision("highest")`（TF32 关），并在 `benchmark.py` 里显式设置。

- 立刻可以算出的预期值（Step 2 的落地）

用第 4 节的 FLOPs 除以上面的 P，**B=4，ctx=512，fp32，TF32 关**：

| 模型 | forward (ms) | backward (ms) | forward+backward (ms) |
|---|---|---|---|
| small | 16.0 | 32.0 | 48.0 |
| medium | 53.7 | 107.4 | 161.1 |
| large | 123 | 246 | 369 |

仅 forward 的大模型（B=4，ctx=512，no_grad）：

| 模型 | fp32 (ms) | bf16 (ms) |
|---|---|---|
| xl | 253 | 88 |
| 2.7B | 424（显存约 12.7 GiB，贴边，不建议） | 148 |

optimizer step 下限（28 B/参数 ÷ 769 GB/s）：small 约 4.7 ms，medium 约 15.4 ms。

**这些都是"理想上限"，也就是时间的下界**，实测应该更慢。原因是模型里的矩阵乘比 8192³ 小得多，效率达不到平台值。可以用你刚测的规模曲线估计：small 的 FFN matmul（2048×768 乘 768×3072）约 9.7 GFLOP，相当于 n≈1700 的方阵，对应曲线上 n=1024 到 2048 附近，fp32 约为平台值的 75% 到 90%，bf16 约为 80% 到 97%。所以：
- small 的 t_meas / t_pred 预期在 1.2 到 2 之间；
- 如果 small 在 ctx=128 时远高于 2，那是 kernel 发射开销，不是 matmul 慢。

## 七、下一步

1. 跑 `count_params.py`，用 meta 设备核对 small、medium、large、xl、2.7B 的参数量。
2. 写 `benchmark.py`，先只跑 `small / ctx=128 / fp32 / TF32 关`。
3. 把实测的 forward、backward、optimizer 三段时间和对应预期贴给我（ctx=128 的预期用 `flops_estimate.py` 算，或者按 ctx=512 的数值换算：线性项除以 4，注意力项除以 16）。

### Step 2：用手算建立预期

一个训练步的 FLOPs 约为 **6 × 参数量（不含 embedding）× token 数**（forward 占 1/3，backward 占 2/3）。

例：small，batch=4，ctx=512。非 embedding 参数约 85M，token 数 = 2048，所以约 1 TFLOP/步。除以 Step 1 测出的实际吞吐，就是你"应该"看到的步时间。

实测和这个预期差 2 倍以内就算正常。差得很远，说明脚本有问题，或者被 launch 开销、非 matmul 算子拖慢，这正是后面要分析的内容。



## 三、具体怎么做（按顺序，做完一格再填一格）

### Step 3：写 `scripts/benchmark.py`

最简骨架（构造函数参数以你仓库为准）：

```python
import timeit, torch
import torch.cuda.nvtx as nvtx
from cs336_basics.model import BasicsTransformerLM
from cs336_basics.optimizer import AdamW
from cs336_basics.nn_utils import cross_entropy

CFG = {  # d_model, d_ff, layers, heads
  "small":  (768, 3072, 12, 12),
  "medium": (1024, 4096, 24, 16),
  "large":  (1280, 5120, 36, 20),
  "xl":     (1600, 6400, 48, 25),
  "2.7B":   (2560, 10240, 32, 32),
}
V = 10000

def run(name, B, ctx, warmup, n):
    d, ff, L, H = CFG[name]
    model = BasicsTransformerLM(vocab_size=V, context_length=ctx, d_model=d,
        num_layers=L, num_heads=H, d_ff=ff, rope_theta=10000.0).cuda()
    opt = AdamW(model.parameters(), lr=1e-4)
    x = torch.randint(0, V, (B, ctx), device="cuda")
    y = torch.randint(0, V, (B, ctx), device="cuda")

    def step():
        torch.cuda.synchronize(); t0 = timeit.default_timer()
        with nvtx.range("forward"):
            loss = cross_entropy(model(x).view(-1, V), y.view(-1))
        torch.cuda.synchronize(); t1 = timeit.default_timer()
        with nvtx.range("backward"):
            loss.backward()
        torch.cuda.synchronize(); t2 = timeit.default_timer()
        with nvtx.range("optimizer"):
            opt.step(); opt.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); t3 = timeit.default_timer()
        return t1 - t0, t2 - t1, t3 - t2

    for _ in range(warmup): step()
    torch.cuda.reset_peak_memory_stats()
    times = torch.tensor([step() for _ in range(n)]) * 1000  # ms
    return times.mean(0), times.std(0), torch.cuda.max_memory_allocated() / 2**30
```

要点：
- 一个循环里同时拿到 forward、backward、optimizer 三段时间，不用分开跑。
- 每段前后都要 `synchronize`，否则测到的只是 CPU 发射 kernel 的时间。
- 只测 forward 的行（xl、2.7B）：把 forward 放进 `torch.no_grad()`，不做 backward 和 step。

### Step 4：先只跑一格

先只跑 `small / ctx=128`，检查：
- loss 是不是有限值
- 三段时间的比例是否合理（通常 backward 约为 forward 的 2 倍）
- 和 Step 2 的预期差多少

这一格对了，再扫满 3.2 的表。

### Step 5：做 3.3（warmup）

同一个脚本，只改 `warmup` 参数。你会看到前几步明显更慢，这就是 CUDA 初始化、allocator 和 kernel 首次加载的代价。

### Step 6：做 3.4（nsys）

```bash
nsys profile -o small_512 --trace=cuda,nvtx,osrt \
  python scripts/benchmark.py --model small --ctx 512 --warmup 5 --steps 3
nsys stats --report cuda_gpu_kern_sum small_512.nsys-rep
```

- 步数设少一点（3 步），否则 profile 文件太大。
- 作业要求给 attention 内部（scores、softmax、加权求和）也打标。做法是写一个带 `nvtx.range` 的 `scaled_dot_product_attention`，替换掉 `cs336_basics.model` 里的同名函数。这就是为什么要读懂 model.py。
- 在 nsys 里按 NVTX 区间过滤，分别看 forward 和 backward 里哪个 kernel 最耗时。

### Step 7：做 3.5 和 3.8（混合精度与显存）

- 混合精度：把 forward 和 loss 包在 `torch.autocast("cuda", dtype=torch.bfloat16)` 里，对比 fp32 的时间和显存。
- 显存：`torch.cuda.memory._record_memory_history()` 开启，跑完 `_dump_snapshot("x.pickle")`，上传到 pytorch.org/memory_viz 看。重点区分"静态"（参数、梯度、Adam 状态）和"随 ctx 增长"（激活值）。

## 四、你文档里需要更正的一处

3.2 的预填预期写的是"xl 仅 fwd fp32 约 4×4B=16GB 超显存"。xl 约 1.6B 参数，fp32 是 **1.6B × 4B ≈ 6.4GB**，在 `no_grad` 下做 forward 是放得下的，预期应该是 OK。2.7B 的 fp32 约 10.4GB，勉强能放，bf16 约 5.2GB 更稳。

注意区分：只做 forward 且 `no_grad` 时不保存激活值；如果 forward 之后要接 backward，激活值会存下来，显存会高得多。

## 五、建议的下一步

按上面 Step 1 到 Step 4 先做完，把 `small / ctx=128` 那一格的结果和你的手算预期贴给我，我可以帮你判断数字是否正常，再决定下一步扫哪些配置。